In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import importlib
from otp_client import get_stops_by_bbox_query
from collections import Counter
import re

In [2]:
data_dir = "/home/simpal/O/TU_Rejseplan/Data/TU/"
tu_stations = pd.read_excel(data_dir + "Stationer_tudatabase.xlsx")

In [25]:
import tu_gtfs_stations_match
importlib.reload(tu_gtfs_stations_match)
from tu_gtfs_stations_match import match_tu_gtfs_stations
tu_gtfs_match_df = match_tu_gtfs_stations(tu_stations,
                       bbox_buffer_m=1000,
                       period=(19725, 20086),
                       name_match_threshold = 0.8)
tu_gtfs_match_df = tu_gtfs_match_df[["tu_station_name", "gtfs_station_id"]]


In [26]:
tu_gtfs_match_df

,tu_station_id,tu_station_name,gtfs_station_id,gtfs_station_name,mode,name_similarity,distance_degree
0,0,Aksel Møllers Have,1:20240102_000008603342,Aksel Møllers Have St. (Metro),SUBWAY,100.0,0.000078
1,1,Albertslund,1:20240102_000008600621,Albertslund St.,S_TRAIN,100.0,0.000339
2,2,Alken,1:20240102_000008600260,Alken St.,RAIL,100.0,0.000659
3,3,Allerød,1:20240102_000008600681,Allerød St.,S_TRAIN,100.0,0.000903
4,4,Amager Strand,1:20240102_000008603324,Amager Strand St. (Metro),SUBWAY,100.0,0.000261
...,...,...,...,...,...,...,...
509,541,Ålsgårde,1:20240102_000008601633,Ålsgårde St.,RAIL,100.0,0.000314
510,542,Åmarken,1:20240102_000008600763,Åmarken St.,S_TRAIN,100.0,0.000324
511,543,Århus H,1:20240102_000008600053,Aarhus H,RAIL,100.0,0.000333
512,544,Årslev,1:20240102_000008600561,Årslev St.,RAIL,100.0,0.000075


In [22]:
def _normalise_name(name: str) -> str:
    """Lowercase, remove punctuation, collapse whitespace."""
    name = name.lower()
    # Replace Å/å with aa
    name = name.replace("å", "aa")
    # Remove common station type suffixes (case-insensitive)
    pattern = r'\s+(station|st\.|st|metro|s-tog|stog)\s*$|\s*\((station|st\.|st|metro|s-tog|stog)\)\s*$'
    while re.search(pattern, name):
        name = re.sub(pattern, '', name)
    # Keep Danish letters, remove other punctuation
    name = re.sub(r"[^\w\søæå]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name
_normalise_name("Østerport St. (Metro)")

'østerport'

In [ ]:
match_df

In [ ]:
tu_stations

In [6]:
# --- Add Lat/Lon (destination) ---
gdf_dest = gpd.GeoDataFrame(
    tu_stations,
    geometry=gpd.points_from_xy(tu_stations["e"], tu_stations["n"]),
    crs="EPSG:32632"
)
gdf_dest = gdf_dest.to_crs("EPSG:4326")

tu_stations["lon"] = gdf_dest.geometry.x
tu_stations["lat"] = gdf_dest.geometry.y

In [ ]:
def get_response(url, query, variables):
    response = requests.post(
        url,
        json={
            "query": query,
            "variables": variables
        }
    )
    if response.status_code != 200:
        raise Exception(response.text)
    if "errors" in response.json():
        raise Exception(response.json()["errors"])
    return response


In [ ]:
def parse_stops_to_df(response_data, station_lat: float, station_lon: float):
    response_data = response_data.json()
    # Retrieve the list of edges
    data = response_data.get("data", {})

    rows = []
    stops = data.get("stopsByBbox", [])

    for stop in stops:
        routes = stop.get("routes", [])
        route_modes = [r.get("mode") for r in routes if r.get("mode")]
        route_names = [r.get("shortName") for r in routes if r.get("shortName")]

        rows.append({
            "distance_degree": None,  # No distance in bbox response
            "stop_gtfsId": stop.get("gtfsId"),
            "name": stop.get("name"),
            "lat": stop.get("lat"),
            "lon": stop.get("lon"),
            "modes": ", ".join(set(route_modes)),
            "routes": ", ".join(route_names)
        })

    df = pd.DataFrame(rows)
    if df.empty:
        df = pd.DataFrame(columns=["distance_degree", "stop_gtfsId", "name", "lat", "lon", "modes", "routes"])
    if not df.empty:
        # Approximate distance to station
        df["distance_degree"] = np.sqrt((df["lat"] - station_lat)**2 + (df["lon"] - station_lon)**2)
    df = df.sort_values(by="distance_degree")
    return pd.DataFrame(df)

In [ ]:
tu_stations.iloc[212]
station_lat = tu_stations.iloc[212]["lat"]
station_lon = tu_stations.iloc[212]["lon"]

In [ ]:
response_data = get_stops_by_bbox_query(
    lat=station_lat,
    lon=station_lon,
    bbox_buffer_m=300
)
parse_stops_to_df(response_data, station_lat, station_lon)


In [ ]:
def cosine_similarity(s1, s2):
    # Convert strings to character frequency vectors
    vec1 = Counter(s1)
    vec2 = Counter(s2)

    # Calculating cosine similarity
    dot_product = sum(vec1[ch] * vec2[ch] for ch in vec1)
    magnitude1 = np.sqrt(sum(count ** 2 for count in vec1.values()))
    magnitude2 = np.sqrt(sum(count ** 2 for count in vec2.values()))
    res = dot_product / (magnitude1 * magnitude2)
    return(res)

In [ ]:
TU_MODE_TO_GTFS = {
    "stog": "S_TRAIN",
    "metro": "SUBWAY",
    "andettog": "RAIL",
    "letbane": "TRAM",
}

def _normalise_name(name: str) -> str:
    """Lowercase, remove punctuation, collapse whitespace."""
    name = name.lower()
    # Remove common station type suffixes (case-insensitive)
    # Order matters: remove longer patterns first
    name = re.sub(r'\s+(station|st\.|st|metro|s-tog|stog)\s*$', '', name)
    name = re.sub(r'\s*\((station|st\.|st|metro|s-tog|stog)\)\s*$', '', name)
    # Keep Danish letters, remove other punctuation
    name = re.sub(r"[^\w\søæå]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name

In [ ]:
tu_station = tu_stations.iloc[212]
bbox_buffer_m = 400
otp_url = "http://localhost:8080/otp/gtfs/v1"
name_match_threshold = 0.6

In [ ]:
# 1. Active modes for this TU station
active_modes = [
    gtfs_mode
    for tu_col, gtfs_mode in TU_MODE_TO_GTFS.items()
    if tu_station.get(tu_col, 0) == 1
]
active_modes

In [ ]:
response = get_stops_by_bbox_query(
    lat=tu_station["lat"],
    lon=tu_station["lon"],
    bbox_buffer_m=bbox_buffer_m,
    otp_url=otp_url,
)
gtfs_df = parse_stops_to_df(response, tu_station["lat"], tu_station["lon"])
gtfs_df

In [ ]:
# 3. Keep only stops that serve at least one relevant mode
def stop_serves_mode(modes_str: str, mode: str) -> bool:
    return mode in [m.strip() for m in modes_str.split(",")]

relevant_stops = gtfs_df[
    gtfs_df["modes"].apply(
        lambda m: any(stop_serves_mode(m, mode) for mode in active_modes)
    )
].copy()
relevant_stops

In [ ]:
tu_name = str(tu_station.get("statnavn", ""))
tu_name

In [ ]:
result = {}


In [ ]:
mode = "RAIL"
mode_stops = relevant_stops[
    relevant_stops["modes"].apply(lambda m: stop_serves_mode(m, mode))
].copy()
mode_stops

In [ ]:
mode_stops["name_similarity"] = mode_stops["name"].apply(
    lambda n: cosine_similarity(_normalise_name("sdklfjsdlfkj"), _normalise_name(n))
)
mode_stops

In [ ]:
mode_stops["name_similarity"] >= name_match_threshold

In [ ]:
name_filtered = mode_stops[mode_stops["name_similarity"] >= name_match_threshold]
name_filtered

In [ ]:
if not name_filtered.empty:
    print(f"  Found {len(name_filtered)} stops for {tu_name} in {mode}")
    mode_stops = name_filtered
else:
    print(
        f"  Warning: no name match (threshold={name_match_threshold}) "
        f"for '{tu_name}' in {mode} stops, using distance only"
    )

# Closest stop by distance
best = mode_stops.loc[mode_stops["distance_degree"].idxmin()]
result[mode] = best


In [ ]:
matches = find_gtfs_stations_for_tu_station(
    tu_station=tu_stations.iloc[7],
    name_match_threshold=0.6,
)

for mode, stop in matches.items():
    print(f"{mode}: {stop['name']} (name similarity: {stop.get('name_similarity', 'N/A')})")

In [ ]:
# for i, row in tu_stations.iterrows():
#     print(i, row["statnavn"])
#     matches = find_gtfs_stations_for_tu_station(
#         tu_station=row,
#         name_match_threshold=0,
#         bbox_buffer_m=bbox_buffer_m,
#     )
#
#     for mode, stop in matches.items():
#         print(f"{mode}: {stop['name']} (name similarity: {stop.get('name_similarity', 'N/A')})")




